In [1]:
!pip install pillow accelerate torchvision

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
import torchvision
import transformers

print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Transformers version:", transformers.__version__)

Torch version: 2.9.1+cu128
Torchvision version: 0.24.1+cu128
Transformers version: 4.57.3


In [3]:
import torch
from transformers import AutoModelForVision2Seq, AutoProcessor

model_id = "Qwen/Qwen2-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Loaded Qwen2 VL 7B")

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
/home5/p319166/.local/lib/python3.11/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded Qwen2 VL 7B


In [4]:
# PNG loader for troubleshooting guides

import os
from pathlib import Path
from PIL import Image
import re

PNG_DIR = "/home5/p319166/TOCAPs/cleaned_png/"

def parse_filename(fname):
    """
    Extracts (tocap_id, page_number) from filenames like:
    DP17-TOCAP12_page_01.png
    """
    m = re.match(r"(.*)_page_(\d+)\.png", fname)
    if not m:
        return None, None
    tocap_id = m.group(1)
    page_num = int(m.group(2))
    return tocap_id, page_num


def load_all_pngs(png_dir=PNG_DIR):
    """
    Loads all PNG files and groups them by TOCAP identifier.
    Returns:
      {
        'DP17-TOCAP12': [
            {'page_num': 1, 'image': PIL.Image},
            {'page_num': 2, 'image': PIL.Image},
            ...
        ],
        ...
      }
    """
    png_dir = Path(png_dir)
    data = {}

    for img_path in png_dir.glob("*.png"):
        fname = img_path.name
        tocap_id, page_num = parse_filename(fname)
        if tocap_id is None:
            print(f"Skipping non-matching filename: {fname}")
            continue

        img = Image.open(img_path).convert("RGB")

        if tocap_id not in data:
            data[tocap_id] = []

        data[tocap_id].append({
            "page_num": page_num,
            "image": img
        })

    # Sort pages inside each TOCAP
    for key in data:
        data[key] = sorted(data[key], key=lambda x: x["page_num"])

    return data


# Load data
tocap_data = load_all_pngs()
print("Loaded TOCAPs:", list(tocap_data.keys()))

Loaded TOCAPs: ['DP17-TOCAP15', 'DP17-TOCAP26', 'DP17-TOCAP17', 'DP17-TOCAP4', 'DP17-TOCAP12', 'DP17-TOCAP6', 'DP17-TOCAP27', 'DP17-TOCAP16', 'DP17-TOCAP28', 'DP17-TOCAP7', 'DP17-TOCAP19', 'DP17-TOCAP18']


In [6]:
import torch
from transformers import AutoModelForVision2Seq, AutoProcessor
import json
import os
import re

# ZERO SHOT PROMPT
zero_shot_prompt = """
Extract procedural knowledge from this Dutch industrial troubleshooting diagram.
CRITICAL RULES:
Extract ONLY entities that ACTUALLY EXIST in the diagram - DO NOT invent or hallucinate entities
If you cannot see more entities clearly, STOP extracting and close the JSON properly
DO NOT repeat the same entity text multiple times with different IDs
Each entity should have UNIQUE text from a distinct node in the diagram
Use sequential IDs: E1, E2, E3, etc.
Types must be exactly: "Action", "Condition", or "Decision"
Preserve EXACT text from each node, including numbering (e.g., "0)", "4-0")
Capture EVERY arrow as an "isPreceededBy" relation
ENTITY TYPES:
Action: An operation to be performed (e.g., "Start Tocap 4", "Wissel product")
Condition: A condition or state to be verified (e.g., "Controleer product, maatvoering")
Decision: A decision point with branching outcomes (e.g., "4-0 Voldoet de kap aan Q productspecificatie's?")
RELATION TYPE:
isPreceededBy: The target step is preceded by the source step (arrow goes from source to target)
EXPECTED VISUAL STRUCTURE:
- Rounded rectangles at top = Start of procedure
- Nodes with "ja/nee" branches = Decision points
- Rectangular boxes = Conditions to verify or Actions to perform
- Arrows show procedural flow (isPreceededBy relationships)
- Numbers in shapes indicate continuation to another page

EXAMPLES OF VISUAL STRUCTURE:
1. Rounded rectangle: {"id": "E1", "type": "Action", "text": "Start Tocap 4 Oppakken kap van bretslede"}
2. Decision node: {"id": "E2", "type": "Decision", "text": "4-0 Voldoet de kap aan Q productspecificatie's?"}
3. Arrow from E1 to E2: {"source": "E2", "target": "E1", "type": "isPreceededBy", "label": "next"}

OUTPUT FORMAT (example with 3 entities - extract MORE if they exist):
{
"entities": [
{"id": "E1", "type": "Action", "text": "Start Tocap 4 Oppakken kap van bretslede"},
{"id": "E2", "type": "Decision", "text": "4-0 Voldoet de kap aan Q productspecificatie's?"},
{"id": "E3", "type": "Action", "text": "0) Wissel product"},
…
],
"relations": [
{"source": "E2", "target": "E1", "type": "isPreceededBy"},
{"source": "E3", "target": "E2", "type": "isPreceededBy"},
…
]
}
IMPORTANT: Extract all the entities you can see, quality over quantity. YOU MUST OUTPUT ONLY VALID JSON. NO EXPLANATIONS."""


# GENERATION FUNCTION 
def qwen2_generate(image, prompt):
    """
    Works with Qwen2 VL. Extracts only generated tokens (no prompt).
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True)
    
    # Let processor handle conversion for Qwen2 VL
    encoded = processor(
        text=inputs,
        images=[image],
        return_tensors="pt"
    ).to(model.device)
    
    # Store input length BEFORE generation
    input_length = encoded["input_ids"].shape[1]
    
    with torch.no_grad():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=4096,  # Increased to prevent truncation
        )
    
    # Extract only the newly generated tokens (skip input prompt)
    generated_ids = output_ids[0][input_length:]
    out = processor.decode(generated_ids, skip_special_tokens=True)
    
    return out.strip()


# JSON PARSER (FIXED - handles markdown and nested JSON)
def extract_json(text):
    """
    Extract JSON from text, handling markdown code blocks and nested structures.
    """
    if not text:
        return None
    
    # Try to find JSON in markdown code block first
    json_match = re.search(r"\s*(\{.*?)\s*```", text, flags=re.DOTALL)
    if json_match:
        candidate = json_match.group(1)
        # Check if we have balanced braces
        brace_count = candidate.count("{") - candidate.count("}")
        
        if brace_count == 0:
            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                pass
        elif brace_count > 0:
            # Need more closing braces - look after the match
            text_after = text[json_match.end(1):]
            for i, char in enumerate(text_after):
                if char == "}":
                    brace_count -= 1
                    if brace_count == 0:
                        candidate = candidate + text_after[:i+1]
                        try:
                            return json.loads(candidate)
                        except json.JSONDecodeError:
                            # Try to fix incomplete JSON
                            return try_fix_incomplete_json(candidate)
                elif char == "{":
                    brace_count += 1
            # If we never found the end, try to fix what we have
            return try_fix_incomplete_json(candidate)
    
    # Try to find JSON without markdown
    first_brace = text.find("{")
    if first_brace == -1:
        return None
    
    # Count braces to find matching closing brace (not just last one)
    brace_count = 0
    last_brace = -1
    
    for i in range(first_brace, len(text)):
        if text[i] == "{":
            brace_count += 1
        elif text[i] == "}":
            brace_count -= 1
            if brace_count == 0:  # Found matching closing brace
                last_brace = i
                break
    
    if last_brace > first_brace:
        candidate = text[first_brace:last_brace + 1]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            return try_fix_incomplete_json(candidate)
    else:
        # Incomplete JSON - try to extract what we can
        candidate = text[first_brace:]
        return try_fix_incomplete_json(candidate)
    
    return None


def try_fix_incomplete_json(text):
    """
    Try to fix incomplete JSON by closing brackets and removing incomplete entries.
    """
    if not text or text.strip() == "":
        return None
    
    text = text.rstrip()
    
    # Count braces and brackets
    brace_count = text.count("{") - text.count("}")
    bracket_count = text.count("[") - text.count("]")
    
    # Close brackets first
    while bracket_count > 0:
        text += "]"
        bracket_count -= 1
    
    # Close braces
    while brace_count > 0:
        text += "}"
        brace_count -= 1
    
    # Remove trailing commas
    text = re.sub(r',\s*}', '}', text)
    text = re.sub(r',\s*]', ']', text)
    
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Last resort: try to extract entities and relations separately
        return extract_partial_json(text)


def extract_partial_json(text):
    """
    Extract entities and relations even from incomplete JSON using regex.
    """
    result = {"entities": [], "relations": []}
    
    # Try to find entities array
    entities_match = re.search(r'"entities"\s*:\s*\[(.*?)\]', text, flags=re.DOTALL)
    if entities_match:
        entities_text = entities_match.group(1)
        # Extract individual entity objects
        entity_pattern = r'\{\s*"id"\s*:\s*"([^"]+)"\s*,\s*"type"\s*:\s*"([^"]+)"\s*,\s*"text"\s*:\s*"([^"]+)"'
        for match in re.finditer(entity_pattern, entities_text):
            result["entities"].append({
                "id": match.group(1),
                "type": match.group(2),
                "text": match.group(3)
            })
    
    # Try to find relations array
    relations_match = re.search(r'"relations"\s*:\s*\[(.*?)\]', text, flags=re.DOTALL)
    if relations_match:
        relations_text = relations_match.group(1)
        # Extract individual relation objects
        relation_pattern = r'\{\s*"source"\s*:\s*"([^"]+)"\s*,\s*"target"\s*:\s*"([^"]+)"\s*,\s*"type"\s*:\s*"([^"]+)"'
        for match in re.finditer(relation_pattern, relations_text):
            result["relations"].append({
                "source": match.group(1),
                "target": match.group(2),
                "type": match.group(3)
            })
    
    return result if result["entities"] or result["relations"] else None


# MAIN LOOP
SAVE_DIR = "/home5/p319166/qwen2_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

for tocap_id, pages in tocap_data.items():
    print(f"Processing {tocap_id} ({len(pages)} pages)")
    out_pages = []
    
    for page in pages:
        page_num = page["page_num"]
        img = page["image"]
        
        try:
            raw = qwen2_generate(img, zero_shot_prompt)
            parsed = extract_json(raw)
            
            # Normalize relation format (handle both "from/to" and "source/target")
            if parsed and "relations" in parsed:
                for relation in parsed["relations"]:
                    if "from" in relation and "source" not in relation:
                        relation["source"] = relation.pop("from")
                    if "to" in relation and "target" not in relation:
                        relation["target"] = relation.pop("to")
            
            out_pages.append({
                "page": page_num,
                "raw": raw,
                "json": parsed
            })
            
            if parsed:
                entity_count = len(parsed.get("entities", []))
                relation_count = len(parsed.get("relations", []))
                print(f"  Page {page_num} ✓ ({entity_count} entities, {relation_count} relations)")
            else:
                print(f"  Page {page_num} (JSON parse failed)")
                # Debug: show last 200 chars
                if raw:
                    print(f"    Last 200 chars: {raw[-200:]}")
            
        except Exception as e:
            print(f"  Error on page {page_num}: {e}")
            import traceback
            out_pages.append({
                "page": page_num,
                "raw": None,
                "json": None,
                "error": str(e),
                "traceback": traceback.format_exc()
            })
    
    with open(os.path.join(SAVE_DIR, f"{tocap_id}.json"), "w") as f:
        json.dump(out_pages, f, indent=2, ensure_ascii=False)
    
    print(f"Saved {tocap_id}")

print("Done")

Processing DP17-TOCAP15 (2 pages)
  Page 1 ✓ (99 entities, 0 relations)
  Page 2 ✓ (20 entities, 10 relations)
Saved DP17-TOCAP15
Processing DP17-TOCAP26 (2 pages)
  Page 1 ✓ (12 entities, 7 relations)
  Page 2 ✓ (14 entities, 13 relations)
Saved DP17-TOCAP26
Processing DP17-TOCAP17 (3 pages)
  Page 1 ✓ (13 entities, 11 relations)
  Page 2 ✓ (19 entities, 17 relations)
  Page 3 ✓ (30 entities, 29 relations)
Saved DP17-TOCAP17
Processing DP17-TOCAP4 (2 pages)
  Page 1 ✓ (142 entities, 0 relations)
  Page 2 ✓ (10 entities, 9 relations)
Saved DP17-TOCAP4
Processing DP17-TOCAP12 (2 pages)
  Page 1 ✓ (114 entities, 0 relations)
  Page 2 ✓ (7 entities, 6 relations)
Saved DP17-TOCAP12
Processing DP17-TOCAP6 (2 pages)
  Page 1 ✓ (28 entities, 27 relations)
  Page 2 ✓ (9 entities, 7 relations)
Saved DP17-TOCAP6
Processing DP17-TOCAP27 (2 pages)
  Page 1 ✓ (15 entities, 14 relations)
  Page 2 ✓ (9 entities, 5 relations)
Saved DP17-TOCAP27
Processing DP17-TOCAP16 (2 pages)
  Page 1 ✓ (18 entities

In [1]:
import torch
import gc
import os

def clean_all_cache():
    """Clean GPU cache, Python garbage collection, and system cache."""
    
    print("Cleaning caches...")
    
    # Clear PyTorch CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(f"✓ Cleared CUDA cache")
        print(f"  GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"  GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    
    # Force Python garbage collection
    gc.collect()
    print("✓ Cleared Python garbage collection")
    
    # Clear Hugging Face cache (optional - careful with this)
    # Uncomment if you want to clear downloaded models
    # hf_cache = os.path.expanduser("~/.cache/huggingface")
    # print(f"HuggingFace cache location: {hf_cache}")
    
    print("Done cleaning!")

# Run it
clean_all_cache()

Cleaning caches...
✓ Cleared CUDA cache
  GPU memory allocated: 0.00 GB
  GPU memory reserved: 0.00 GB
✓ Cleared Python garbage collection
Done cleaning!
